# MMIP Week 4 — 四題 GPU 實驗

蘇芯儀 / Sharon0726

T4 GPU，依序執行。Rotten Tomatoes RNN/LSTM；CIFAR-10 ViT/ResNet；Flickr8k GRU/LSTM Captioning。包含真實訓練、測試、BLEU、計時與驗證。Gemini 人工評審另外見 REPORT.md。

In [ ]:
import base64, io, os, pathlib, subprocess, sys, zipfile, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
work=pathlib.Path('/content/mmip_week4'); work.mkdir(exist_ok=True)
(work/'results').mkdir(exist_ok=True)
payload='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z: z.extractall(work)
os.chdir(work)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r','requirements.txt'])


In [ ]:
process=subprocess.Popen([sys.executable,'-u','run_all.py'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with open('results/training.log','w') as log:
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line); log.flush()
assert process.wait()==0, 'Training failed; inspect the log.'


In [ ]:
from google.colab import files
out=pathlib.Path('/content/MMIP_Week4_results.zip')
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['results','checkpoints']:
        for p in pathlib.Path(folder).rglob('*'):
            if p.is_file(): z.write(p,p.as_posix())
print('Export bytes:',out.stat().st_size)
files.download(str(out))
